In [0]:
#adding src path and reading config file
import sys
sys.path.append("/Workspace/DataStore/DataStore/src")
import yaml
from core.Base import BaseConfig
from core.Read_Yaml import ReadYaml
from core.Create_Tabls import CreateTable
from core.Create_Schem import CreateSchema
from core.TableHandler import TableHandler
from pyspark.sql.functions import current_timestamp
sys.path.append("/Workspace/DataStore/DataStore/src")
path = "/Workspace/DataStore/DataStore/configs/bronze.yaml"
settings_path = "/Workspace/DataStore/DataStore/configs/settings.yaml"


In [0]:
#getting table information and creating table and schema if not exists
table_Name = "Businessunit"
table_info= ReadYaml.read_yaml(path, table_Name)
table_info = BaseConfig(table_info)
config = {
    "spark":spark,  
    "catalog": table_info.catalog,
    "schema":table_info.schema,
    "table_name":table_info.tableName,
    "path":table_info.path,
    "table_type":table_info.table_type
}
CreateSchema.create_schema(spark,table_info.catalog, table_info.schema)
CreateTable.create_table(config=config)

In [0]:
#reading data from source path
source_path = ReadYaml.read_yaml(settings_path, "paths")
source_businessunit = f"{source_path['source']}/Businessunit"
bu_df = spark.read.parquet(source_businessunit)
bu_df.drop("selfUri")
bu_df = bu_df.withColumn("InsertedAt", current_timestamp())

#writing data to the bronze external table
TableHandler.write(df= bu_df, spark=spark,
                    table_name=f"{table_info.catalog}.{table_info.schema}.{table_info.tableName}",
                     mergeSchema=True,
                      mode="overwrite")



In [0]:
%sql
select * from datastore.bronze.Businessunit